In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import libsbml

# Add the project root to the Python path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Import AAAIM functions
from core import annotate_model, curate_model, model_info

In [2]:
model_file = "./test_models/MODEL1007200000.xml"
entity_type = "chemicals"
species_ids = model_info.get_all_species_ids(model_file, entity_type)
print(species_ids)
model_info.extract_model_info(model_file, species_ids, entity_type)

['b_met', 'b_gly', 'b_ser', 'GAR', 'NADPH', 'BET', 'DUMP', 'Fol', 'CO', 'HCHO', 'c_thf', 'm_thf', 'c_5mf', 'c_2cf', 'c_1cf', 'c_10f', 'dhf', 'm_2cf', 'm_1cf', 'm_10f', 'aic', 'c_gly', 'hcy', 'c_ser', 'sah', 'sam', 'met', 'c_coo', 'm_ser', 'm_gly', 'm_coo', 'src', 'dmg']


{'model_name': 'Nijhout2006_Hepatic_Folate_Metab',
 'model_type': <ModelType.SBML: 'SBML'>,
 'format_info': {'has_fbc': False,
  'has_qual': False,
  'num_species': 33,
  'num_reactions': 35},
 'display_names': {'c_thf': '',
  'm_gly': 'mit_Glycine',
  'm_1cf': 'm_5-10-methenyl-THF',
  'm_thf': '',
  'src': 'Sarcosine',
  'dhf': '',
  'NADPH': '',
  'aic': 'AICAR',
  'm_2cf': 'm_5-10-methylene-THF',
  'c_gly': 'c_Glycine',
  'sah': 'S-adenosylhomocysteine',
  'c_ser': '',
  'CO': 'CO2',
  'met': 'methionine',
  'GAR': '',
  'b_met': '',
  'BET': 'Betaine',
  'sam': 'S-adenosylmethionine',
  'm_ser': 'm_Serine',
  'Fol': 'Folate',
  'c_10f': 'c_10-formyl-THF',
  'm_10f': 'm_10-formyl-THF',
  'dmg': 'Dimethylglycine',
  'b_gly': '',
  'c_5mf': 'c_5-methyl-THF',
  'c_coo': 'c_formate',
  'HCHO': 'Formaldehyde',
  'c_1cf': 'c_5-10-methenyl-THF',
  'hcy': 'Homocysteine',
  'DUMP': 'dUMP',
  'c_2cf': 'c_5-10-methylene-THF',
  'b_ser': '',
  'm_coo': 'm_formate'},
 'reactions': ['$b_gly -> c_

In [3]:
# using the free openrouter api
result = annotate_model(
    model_file="./test_models/MODEL1007200000.xml",
    # llm_model="gpt-5-mini-2025-08-07",
    llm_model="openrouter/free",
    entity_type='chemical',
    database='chebi',
    method="rag",
    top_k=3,
)

LLM Reason: The model describes hepatic folate metabolism with cytosolic (c_) and mitochondrial (m_) compartments. Boundary species (b_) represent blood/input pools of methionine, glycine, serine. Display names from the model (e.g., "mit_Glycine", "S-adenosylhomocysteine", "CO2") and reaction context (e.g., GAR consumption in purine synthesis, NADPH redox role, dUMP in thymidylate cycle) were used to map each abbreviation to its biochemical entity. Standardized ChEBI names and common synonyms are provided, prioritizing the most specific and widely used terms. Compartment prefixes (c_, m_) do not alter the chemical identity, so cytosolic and mitochondrial folate forms share the same base names (e.g., THF, 5,10-methylene-THF).
Recommendations saved to MODEL1007200000.xml_recommendations.csv


In [ ]:
# gpt 5 mini
result = annotate_model(
    model_file="./test_models/MODEL1007200000.xml",
    llm_model="gpt-5-mini-2025-08-07",
    entity_type='chemical',
    database='chebi',
    method="rag",
    top_k=3,
)

LLM Reason: Mapping is based on the model display names and common biochemical synonyms. Cytosolic (c_) and mitochondrial (m_) prefixes were ignored per instruction to give general chemical names. Examples: Fol was labeled "Folate" in the model → folate/folic acid; c_thf/m_thf correspond to tetrahydrofolate (THF) and the various one-carbon substituted THF forms are given as 5-methyl, 5,10-methylene, 5,10-methenyl, and 10-formyl derivatives per the display names; DUMP is dUMP (deoxyuridine monophosphate); GAR corresponds to glycinamide ribonucleotide (GAR) in purine biosynthesis; AICAR is the 5-aminoimidazole-4-carboxamide ribonucleotide; NADPH, BET (betaine), SRC (sarcosine), DMG (dimethylglycine), SAM and SAH, MET, HCY, GLY and SER are standard metabolites named as shown in the model. CO was mapped to carbon dioxide (model listed CO as CO2) and HCHO to formaldehyde per display names. Formate/formic acid used for c_coo/m_coo and m_coo as the model lists these as formate.
Recommendation